# Autoencoder for Anomaly Detection

This notebook explores a different approach to the same problem using an autoencoder:

- **Goal:** Detect abnormal segments in the signal by learning a reconstruction model.
- **Approach:**
  - Train an autoencoder to reconstruct signal segments.
  - Train **only on healthy (normal) data** to capture typical patterns.
  - Define a threshold (e.g., 95th percentile of reconstruction error on healthy data) to detect anomalies.
  - Evaluate on both healthy and unhealthy segments.
- **Motivation:** 
  - Faulty segments may indicate subtle changes in the baby's state.
  - By focusing on reconstruction error, we may detect segments that deviate from normal behavior without explicit labels.


In [1]:
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import os

from helpers.zero_handlers import interpolate_zeros
from helpers.dataset import WindowDataset

from models.definitions import WindowAE
from models.functions import ae_loss, train, get_signal_recon_error

In [ ]:
# Config 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WINDOW_SIZE = 10
STRIDE = 1
BATCH_SIZE = 128
EPOCHS = 5
LATENT_DIM = 4
LEARNING_RATE = 1e-3

# Evaluation
RECON_THRESHOLD_PERCENTILE = 95
TEST_SIZE = 0.3

In [3]:
# Load and Split All Data
print("Loading and splitting all signals")
df = pd.read_csv("data/hea.csv", index_col=0)

all_signals_by_stage = {} # New data structure
skipped = 0
DATABASE_PATH = "data/zero_quantiles/0.15_data/*.dat"

for dat_file in tqdm(glob.glob(DATABASE_PATH), desc="Loading and splitting files"):
    rec_id = int(os.path.basename(dat_file).split('.')[0])
    
    try:
        label = df[df["metadata.id"] == rec_id]["fetal_state"].iloc[0]
        cuttoff_in_min = df[df["metadata.id"] == rec_id]["stage_2_index"].iloc[0]
    except IndexError:
        skipped += 1
        continue
    
    raw_data = np.fromfile(dat_file, dtype=np.int16)
    fhr = (raw_data[0::2] - 0.0) / 100.0 # fhr_gain/baseline
    uc = raw_data[1::2]
    
    cuttoff = int(cuttoff_in_min * 60 * 4) # 60s/min, 4Hz
    
    # Ensure cutoff is valid and doesn't create empty arrays
    if cuttoff <= WINDOW_SIZE or (len(fhr) - cuttoff) <= WINDOW_SIZE:
            skipped += 1
            continue
            
    # Preprocess and store Stage 1
    fhr_s1 = fhr[:-cuttoff]
    uc_s1 = uc[:-cuttoff]
    fhr_s1 = interpolate_zeros(fhr_s1)
    uc_s1 = interpolate_zeros(uc_s1)
    
    # Preprocess and store Stage 2
    fhr_s2 = fhr[-cuttoff:]
    uc_s2 = uc[-cuttoff:]
    fhr_s2 = interpolate_zeros(fhr_s2)
    uc_s2 = interpolate_zeros(uc_s2)

    # Check for valid data
    if (len(fhr_s1) < WINDOW_SIZE or np.std(fhr_s1) < 1e-6 or np.std(uc_s1) < 1e-6 or
        len(fhr_s2) < WINDOW_SIZE or np.std(fhr_s2) < 1e-6 or np.std(uc_s2) < 1e-6):
        skipped += 1
        continue

    # Standardize stages independently
    fhr_s1 = StandardScaler().fit_transform(fhr_s1.reshape(-1, 1)).flatten()
    uc_s1 = StandardScaler().fit_transform(uc_s1.reshape(-1, 1)).flatten()
    fhr_s2 = StandardScaler().fit_transform(fhr_s2.reshape(-1, 1)).flatten()
    uc_s2 = StandardScaler().fit_transform(uc_s2.reshape(-1, 1)).flatten()
    
    all_signals_by_stage[rec_id] = {
        "label": label,
        "stage_1": {"fhr": fhr_s1, "uc": uc_s1, "label": label, "id": rec_id},
        "stage_2": {"fhr": fhr_s2, "uc": uc_s2, "label": label, "id": rec_id}
    }

print(f"Loaded and split {len(all_signals_by_stage)} signals (skipped {skipped} invalid signals)")

[1/N] Loading and splitting all signals...


Loading and splitting files: 100%|██████████| 228/228 [00:00<00:00, 458.11it/s]

Loaded and split 204 signals (skipped 24 invalid signals)


In [ ]:
# Loop through each stage and run the analysis
for stage_to_process in ["stage_1", "stage_2"]:
    print("="*80)
    print(f"STARTING ANALYSIS FOR: {stage_to_process.upper()}")
    print("="*80)

    # Select Data for this Stage
    print(f"\n[2/7] Selecting signals for {stage_to_process}...")
    
    # Build the 'all_signals' list just for this stage
    all_signals_this_stage = {}
    for rec_id, data in all_signals_by_stage.items():
        all_signals_this_stage[rec_id] = data[stage_to_process]
    
    print(f"Total signals for this stage: {len(all_signals_this_stage)}")
        
    #  Split Data into Train/Test sets 
    print("\n[3/7] Splitting data into training and testing sets...")
    
    normal_signals = {k: v for k, v in all_signals_this_stage.items() if v["label"] == "normal"}
    abnormal_signals = {k: v for k, v in all_signals_this_stage.items() if v["label"] == "abnormal"}

    if len(normal_signals) < 2:
        print(f"Not enough normal signals for {stage_to_process} to split. Skipping stage.")
        continue
        
    normal_ids = list(normal_signals.keys())
    normal_train_ids, normal_test_ids = train_test_split(normal_ids, test_size=TEST_SIZE, random_state=42)

    normal_train_signals = [normal_signals[i] for i in normal_train_ids]
    
    test_signals_dict = {i: normal_signals[i] for i in normal_test_ids}
    test_signals_dict.update(abnormal_signals)
    test_signals = list(test_signals_dict.values()) # This is now a list of dicts

    print(f"Training set size: {len(normal_train_signals)} (normal signals)")
    print(f"Testing set size: {len(test_signals)} ({len(normal_test_ids)} normal, {len(abnormal_signals)} abnormal)")

    # Train AE model 
    print(f"\n[4/7] Training AE model for {stage_to_process}...")
    
    if not normal_train_signals:
        print("No 'normal' signals available for training. Skipping stage.")
        continue

    train_dataset = WindowDataset(normal_train_signals, WINDOW_SIZE, STRIDE)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    print(f"Extracted {len(train_dataset)} windows for training.")
    if len(train_dataset) == 0:
        print("No windows extracted for training. Skipping stage.")
        continue
    
    model = WindowAE(window_size=WINDOW_SIZE, in_channels=2, latent_dim=LATENT_DIM).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    train(train_loader, model, optimizer, EPOCHS, DEVICE)
    
    os.makedirs("results/ae", exist_ok=True)
    model_path = f"results/ae/window_ae_{stage_to_process}.pth"
    torch.save({'model_state_dict': model.state_dict(), 'window_size': WINDOW_SIZE, 'latent_dim': LATENT_DIM, 'stage': stage_to_process}, model_path)
    print(f"Model saved to {model_path}")

    # Find Anomaly Threshold 
    print(f"\n[5/7] Finding anomaly threshold from {len(normal_test_ids)} normal test signals...")
    
    normal_test_signals_list = [normal_signals[i] for i in normal_test_ids]
    normal_errors = []
    
    for signal_data in tqdm(normal_test_signals_list, desc="Calculating normal errors"):
        error = get_signal_recon_error(model, signal_data, WINDOW_SIZE, STRIDE, BATCH_SIZE, DEVICE)
        normal_errors.append(error)
    
    if not normal_errors:
        print("Warning: No normal test signals to build threshold. Using 0.")
        threshold = 0
    else:
        threshold = np.percentile(normal_errors, RECON_THRESHOLD_PERCENTILE)
    
    print(f"Anomaly Threshold ({RECON_THRESHOLD_PERCENTILE}th percentile): {threshold:.6f}")
    

    # Evaluate Model 
    print(f"\n[6/7] Evaluating on the test set ({len(test_signals)} signals)...")
    y_true = []
    y_pred = []
    
    for signal_data in tqdm(test_signals, desc="Testing"):
        error = get_signal_recon_error(model, signal_data, WINDOW_SIZE, STRIDE, BATCH_SIZE, DEVICE)
        is_abnormal = error > threshold
        
        y_pred.append(1 if is_abnormal else 0)
        y_true.append(1 if signal_data["label"] == "abnormal" else 0)
    
    # Results
    print("\n" + "="*60)
    print(f"RESULTS FOR {stage_to_process.upper()}")
    print("="*60)
    
    if not y_true:
        print("No signals in the test set to evaluate.")
    else:
        print(classification_report(y_true, y_pred, target_names=['Normal', 'Abnormal'], digits=4, zero_division=0))
    
    print("\nConfiguration:")
    print(f"  Stage processed: {stage_to_process}")
    print(f"  Model Type: Autoencoder (AE) with Reconstruction Loss")
    print(f"  Anomaly Threshold: {RECON_THRESHOLD_PERCENTILE}th percentile of normal test error")
    
    print("\nHypothesis:")
    print(f"  Abnormal signals should have a reconstruction error")
    print(f"  higher than most normal signals (error > {threshold:.6f}).")
    print("\n")

print("="*80)
print("All stages processed.")
print("="*80)

STARTING ANALYSIS FOR: STAGE_1

[2/7] Selecting signals for stage_1...
Total signals for this stage: 204

[3/7] Splitting data into training and testing sets...
Training set size: 91 (normal signals)
Testing set size: 113 (40 normal, 73 abnormal)

[4/7] Training AE model for stage_1...
Extracted 1309641 windows for training.
Training on 1309641 windows...


Epoch 1/5: 100%|██████████| 10232/10232 [00:20<00:00, 495.02it/s]


  Reconstruction Loss: 0.0334


Epoch 2/5: 100%|██████████| 10232/10232 [00:19<00:00, 514.19it/s]


  Reconstruction Loss: 0.0096


Epoch 3/5: 100%|██████████| 10232/10232 [00:20<00:00, 487.59it/s]


... (19 outputs left out)

Calculating normal errors: 100%|██████████| 40/40 [00:01<00:00, 25.58it/s]


Anomaly Threshold (95th percentile): 0.287434

[6/7] Evaluating on the test set (113 signals)...


Testing: 100%|██████████| 113/113 [00:03<00:00, 29.36it/s]


RESULTS FOR STAGE_2
              precision    recall  f1-score   support

      Normal     0.3455    0.9500    0.5067        40
    Abnormal     0.3333    0.0137    0.0263        73

    accuracy                         0.3451       113
   macro avg     0.3394    0.4818    0.2665       113
weighted avg     0.3376    0.3451    0.1964       113


Configuration:
  Stage processed: stage_2
  Model Type: Autoencoder (AE) with Reconstruction Loss
  Anomaly Threshold: 95th percentile of normal test error

Hypothesis:
  Abnormal signals should have a reconstruction error
  higher than most normal signals (error > 0.287434).


All stages processed.
